# Smart AI Assistant with Human Feedback (Colab)

This notebook is a Colab-ready version of the assignment project.

It covers the full rubric:
- Generate 2-3 text responses with GPT-2
- Collect human feedback through ratings and a best-response choice
- Rank responses to simulate an RLHF reward model
- Generate an image from the same prompt with Stable Diffusion


## How to use this notebook

1. Enable GPU in Colab if possible.
2. Run the install cell.
3. Run the setup cell.
4. Set a shared prompt.
5. Generate text and image outputs.
6. Edit the ratings cell and choose the best response.
7. Run the ranking cell to simulate RLHF.


In [ ]:
# Install project dependencies for Colab
!pip -q install torch transformers diffusers accelerate safetensors pillow pandas


In [ ]:
from __future__ import annotations

import re
import sqlite3
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
from typing import Any

import pandas as pd
import torch
from IPython.display import display
from diffusers import StableDiffusionPipeline
from transformers import AutoModelForCausalLM, AutoTokenizer


def detect_device(torch_module: Any) -> str:
    if torch_module.cuda.is_available():
        return "cuda"
    if getattr(torch_module.backends, "mps", None) and torch_module.backends.mps.is_available():
        return "mps"
    return "cpu"


def normalize_generated_text(text: str) -> str:
    cleaned = text.replace("\r", " ").strip()
    cleaned = re.sub(r"\s+", " ", cleaned)
    return cleaned.lstrip(":,- ")


@dataclass(frozen=True)
class GeneratedResponse:
    id: int
    prompt: str
    response_text: str


class FeedbackStore:
    def __init__(self, database_path: str | Path) -> None:
        self.database_path = Path(database_path)
        self.database_path.parent.mkdir(parents=True, exist_ok=True)
        self._initialize_database()

    def _connect(self) -> sqlite3.Connection:
        connection = sqlite3.connect(self.database_path)
        connection.row_factory = sqlite3.Row
        connection.execute("PRAGMA foreign_keys = ON")
        return connection

    def _initialize_database(self) -> None:
        with self._connect() as connection:
            connection.executescript(
                """
                CREATE TABLE IF NOT EXISTS responses (
                    id INTEGER PRIMARY KEY AUTOINCREMENT,
                    prompt TEXT NOT NULL,
                    response_text TEXT NOT NULL,
                    created_at TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP
                );

                CREATE TABLE IF NOT EXISTS feedback (
                    id INTEGER PRIMARY KEY AUTOINCREMENT,
                    response_id INTEGER NOT NULL,
                    rating INTEGER NOT NULL CHECK (rating BETWEEN 1 AND 5),
                    selected_best INTEGER NOT NULL DEFAULT 0 CHECK (selected_best IN (0, 1)),
                    created_at TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP,
                    FOREIGN KEY (response_id) REFERENCES responses(id) ON DELETE CASCADE
                );
                """
            )

    def save_generated_responses(self, prompt: str, responses: list[str]) -> list[GeneratedResponse]:
        created = []
        with self._connect() as connection:
            cursor = connection.cursor()
            for response_text in responses:
                if not response_text.strip():
                    continue
                cursor.execute(
                    "INSERT INTO responses (prompt, response_text) VALUES (?, ?)",
                    (prompt, response_text.strip()),
                )
                created.append(
                    GeneratedResponse(
                        id=int(cursor.lastrowid),
                        prompt=prompt,
                        response_text=response_text.strip(),
                    )
                )
        return created

    def record_feedback(self, ratings_by_response_id: dict[int, int], best_response_id: int | None) -> None:
        with self._connect() as connection:
            cursor = connection.cursor()
            for response_id, rating in ratings_by_response_id.items():
                if rating < 1 or rating > 5:
                    raise ValueError("Ratings must be between 1 and 5.")
                cursor.execute(
                    "INSERT INTO feedback (response_id, rating, selected_best) VALUES (?, ?, ?)",
                    (response_id, rating, 1 if response_id == best_response_id else 0),
                )

    def get_ranked_responses(self, prompt: str) -> pd.DataFrame:
        with self._connect() as connection:
            rows = connection.execute(
                """
                SELECT
                    responses.id AS response_id,
                    responses.response_text,
                    COALESCE(AVG(feedback.rating), 0.0) AS average_rating,
                    COUNT(feedback.rating) AS rating_count,
                    COALESCE(SUM(feedback.selected_best), 0) AS best_vote_count
                FROM responses
                LEFT JOIN feedback ON feedback.response_id = responses.id
                WHERE responses.prompt = ?
                GROUP BY responses.id
                ORDER BY best_vote_count DESC, average_rating DESC, rating_count DESC, responses.id ASC
                """,
                (prompt,),
            ).fetchall()

        records = []
        for row in rows:
            reward_proxy = round((float(row["average_rating"]) + 2.0 * int(row["best_vote_count"])), 2)
            records.append(
                {
                    "Response ID": int(row["response_id"]),
                    "Average Rating": round(float(row["average_rating"]), 2),
                    "Ratings": int(row["rating_count"]),
                    "Best Votes": int(row["best_vote_count"]),
                    "Reward Proxy": reward_proxy,
                    "Response Text": str(row["response_text"]),
                }
            )
        return pd.DataFrame(records)


class TextGenerationService:
    def __init__(self, model_id: str) -> None:
        self.model_id = model_id
        self.device = detect_device(torch)
        self.tokenizer = AutoTokenizer.from_pretrained(model_id)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        model_kwargs = {}
        if self.device == "cuda":
            model_kwargs["torch_dtype"] = torch.float16

        self.model = AutoModelForCausalLM.from_pretrained(model_id, **model_kwargs)
        self.model.to(self.device)
        self.model.eval()

    def generate_responses(self, prompt: str, response_count: int = 3, max_new_tokens: int = 90) -> list[str]:
        prompt_template = f"Prompt: {prompt}\nAssistant:"
        encoded = self.tokenizer(prompt_template, return_tensors="pt")
        encoded = {key: value.to(self.device) for key, value in encoded.items()}
        prompt_length = encoded["input_ids"].shape[1]

        candidate_responses = []
        sampling_temperatures = [0.7, 0.9, 1.1, 0.8, 1.0, 1.2]

        for index in range(max(response_count * 2, 6)):
            seed = 42 + index
            torch.manual_seed(seed)
            if self.device == "cuda":
                torch.cuda.manual_seed_all(seed)

            output_ids = self.model.generate(
                **encoded,
                do_sample=True,
                max_new_tokens=max_new_tokens,
                temperature=sampling_temperatures[index % len(sampling_temperatures)],
                top_p=0.92,
                repetition_penalty=1.15,
                pad_token_id=self.tokenizer.eos_token_id,
                eos_token_id=self.tokenizer.eos_token_id,
                num_return_sequences=1,
            )

            decoded = self.tokenizer.decode(output_ids[0][prompt_length:], skip_special_tokens=True)
            cleaned = normalize_generated_text(decoded)
            if cleaned and cleaned not in candidate_responses:
                candidate_responses.append(cleaned)
            if len(candidate_responses) >= response_count:
                break

        return candidate_responses[:response_count]


class ImageGenerationService:
    def __init__(self, model_id: str) -> None:
        self.model_id = model_id
        self.device = detect_device(torch)
        dtype = torch.float16 if self.device == "cuda" else torch.float32
        self.pipeline = StableDiffusionPipeline.from_pretrained(model_id, torch_dtype=dtype)
        if self.device in {"cuda", "mps"}:
            self.pipeline = self.pipeline.to(self.device)
        else:
            self.pipeline.enable_attention_slicing()

    def generate_image(self, prompt: str, num_inference_steps: int = 30, guidance_scale: float = 7.5):
        result = self.pipeline(
            prompt=prompt,
            num_inference_steps=num_inference_steps,
            guidance_scale=guidance_scale,
        )
        return result.images[0]


def save_generated_image(image, output_dir: str | Path, prompt: str) -> Path:
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    slug = re.sub(r"[^a-zA-Z0-9]+", "-", prompt.lower()).strip("-")[:40] or "prompt"
    timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
    image_path = output_dir / f"{timestamp}-{slug}.png"
    image.save(image_path)
    return image_path


DATA_DIR = Path("project_outputs")
DB_PATH = DATA_DIR / "feedback.db"
IMAGE_DIR = DATA_DIR / "generated_images"

store = FeedbackStore(DB_PATH)
print(f"Using device: {detect_device(torch)}")
print(f"Feedback database: {DB_PATH}")


In [ ]:
# Shared prompt and model configuration
PROMPT = "A robot helping humans in daily life"
TEXT_MODEL_ID = "gpt2"
IMAGE_MODEL_ID = "runwayml/stable-diffusion-v1-5"
TEXT_RESPONSE_COUNT = 3
TEXT_MAX_NEW_TOKENS = 90
IMAGE_NUM_INFERENCE_STEPS = 30
IMAGE_GUIDANCE_SCALE = 7.5

print("Prompt:", PROMPT)
print("Text model:", TEXT_MODEL_ID)
print("Image model:", IMAGE_MODEL_ID)
print("If Stable Diffusion access fails, keep the workflow the same and replace IMAGE_MODEL_ID with another public Stable Diffusion checkpoint.")


In [ ]:
# Generate text responses and one image from the same prompt
text_service = TextGenerationService(TEXT_MODEL_ID)
image_service = ImageGenerationService(IMAGE_MODEL_ID)

text_responses = text_service.generate_responses(
    prompt=PROMPT,
    response_count=TEXT_RESPONSE_COUNT,
    max_new_tokens=TEXT_MAX_NEW_TOKENS,
)
generated_responses = store.save_generated_responses(PROMPT, text_responses)

print("Generated text responses:\n")
for index, response in enumerate(generated_responses, start=1):
    print(f"Response {index} (ID {response.id}):")
    print(response.response_text)
    print()

generated_image = image_service.generate_image(
    prompt=PROMPT,
    num_inference_steps=IMAGE_NUM_INFERENCE_STEPS,
    guidance_scale=IMAGE_GUIDANCE_SCALE,
)
generated_image_path = save_generated_image(generated_image, IMAGE_DIR, PROMPT)
display(generated_image)
print(f"Image saved to: {generated_image_path}")


## Human feedback step

Edit the values in the next cell.

- Choose one `BEST_RESPONSE_ID`
- Rate every response from `1` to `5`

This is the simplified human feedback stage used to simulate RLHF.


In [ ]:
# Update these values before running the cell
BEST_RESPONSE_ID = generated_responses[0].id
ratings_by_response_id = {
    generated_responses[0].id: 5,
    generated_responses[1].id: 4,
    generated_responses[2].id: 3,
}

store.record_feedback(ratings_by_response_id, BEST_RESPONSE_ID)
print("Feedback stored successfully.")


In [ ]:
# Show the RLHF-style ranking table
ranking_table = store.get_ranked_responses(PROMPT)
display(ranking_table)
print("Ranking order uses best-response votes, then average rating, then rating count.")
print("This ranking table is the notebook's reward-model simulation.")


## Demo tips

- Use one prompt that works for both text and image generation.
- Show the three generated text candidates first.
- Show the generated image next.
- Explain that feedback is stored in SQLite.
- Run the ranking cell to show the simplified RLHF result.
